## Setup

In [1]:
%%capture
!pip install transformers
!pip install sentencepiece
!pip install seqeval
!pip install datasets

In [2]:
## Mount GDrive
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

## Imports
import os
import sys
import nltk
import time
import torch
import random
import subprocess
import numpy as np
import pandas as pd
import datetime as dt
from itertools import groupby
from tqdm.notebook import tqdm
from datasets import load_dataset
from transformers import pipeline
from collections import Counter, defaultdict
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForTokenClassification, AutoTokenizer
from seqeval.metrics import f1_score as seq_f1, precision_score as seq_precision, recall_score as seq_recall, classification_report as seq_classification
from sklearn.metrics import f1_score as skl_f1, precision_score as skl_precision, recall_score as skl_recall, classification_report as skl_classification

Mounted at /content/drive/


In [3]:
# Append the library files into the notebook system path for import
sys.path.append('/content/drive/Shareddrives/Machine Translation/Model benchmarking/Libraries/1.0.2')
# import custom library files
import ner, utils

## Load datasets

### Peoples daily
https://huggingface.co/datasets/peoples_daily_ner

In [4]:
pdaily_label_map = {
    "O": 0,
    "B-PER": 1,
    "I-PER": 2,
    "B-ORG": 3,
    "I-ORG": 4,
    "B-LOC": 5,
    "I-LOC": 6,
}

pdaily = ner.ReadNERData()
pdaily_words, pdaily_labels = pdaily.read_dataset('peoples_daily_ner', pdaily_label_map)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:72: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating test Split


  0%|          | 0/4637 [00:00<?, ?it/s]

In [5]:
print(ner.check_labels(pdaily_labels))
# Dataset Label Map Alignment to LOC, ORG, PERS, MISC

# harem_label_alignment = {
#     'B-COISA': 'O',
#     'B-LOCAL': 'B-LOC',
#     'B-OBRA': 'O',
#     'I-VALOR': 'O',
#     'I-OUTRO': 'O',
#     'B-ABSTRACCAO': 'O',
#     'B-TEMPO': 'O',
#     'O': 'O',
#     'I-TEMPO': 'O',
#     'B-ACONTECIMENTO': 'O',
#     'I-ORGANIZACAO': 'I-ORG',
#     'I-PESSOA': 'I-PER',
#     'B-PESSOA': 'B-PER',
#     'B-VALOR': 'O',
#     'I-ABSTRACCAO': 'O',
#     'B-ORGANIZACAO': 'B-ORG',
#     'I-COISA': 'O',
#     'I-LOCAL': 'I-LOC',
#     'B-OUTRO': 'O',
#     'I-ACONTECIMENTO': 'O',
#     'I-OBRA': 'O',
# }

# Align the dataset labels to the standard labels
# harem_labels = ner.align_dataset(harem_labels, harem_label_alignment)
# print(ner.check_labels(harem_labels))

{'B-ORG', 'O', 'B-LOC', 'I-ORG', 'B-PER', 'I-LOC', 'I-PER'}


### wikiann

In [6]:
wikiann_label_map = {
    "O": 0,
    "B-PER": 1,
    "I-PER": 2,
    "B-ORG": 3,
    "I-ORG": 4,
    "B-LOC": 5,
    "I-LOC": 6
}

wikiann = ner.ReadNERData()
wikiann_words, wikiann_labels = wikiann.read_dataset('wikiann', wikiann_label_map, lang='zh')

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating test Split


  0%|          | 0/10000 [00:00<?, ?it/s]

In [7]:
print(ner.check_labels(wikiann_labels))
# Dataset Label Map Alignment to LOC, ORG, PERS, MISC

{'B-ORG', 'O', 'B-LOC', 'I-ORG', 'B-PER', 'I-LOC', 'I-PER'}


### MRSA
https://huggingface.co/datasets/msra_ner


In [8]:
mrsa_label_map = {
    "O": 0,
    "B-PER": 1,
    "I-PER": 2,
    "B-ORG": 3,
    "I-ORG": 4,
    "B-LOC": 5,
    "I-LOC": 6,
}

mrsa = ner.ReadNERData()
mrsa_words, mrsa_labels = mrsa.read_dataset('msra_ner', mrsa_label_map)

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating test Split


  0%|          | 0/3443 [00:00<?, ?it/s]

# Evaluate model

In [11]:
alignment = {
'O'             : 'O',
'B-CARDINAL'    : 'O',
'B-DATE'        : 'O',
'B-EVENT'       : 'O',
'B-FAC'         : 'O',
'B-GPE'         : 'O',
'B-LANGUAGE'    : 'O',
'B-LAW'         : 'O',
'B-LOC'         : 'B-LOC',
'B-MONEY'       : 'O',
'B-NORP'        : 'O',
'B-ORDINAL'     : 'O',
'B-ORG'         : 'B-ORG',
'B-PERCENT'     : 'O',
'B-PERSON'      : 'B-PER',
'B-PRODUCT'     : 'O',
'B-QUANTITY'    : 'O',
'B-TIME'        : 'O',
'B-WORK_OF_ART' : 'O',
'I-CARDINAL'    : 'O',
'I-DATE'        : 'O',
'I-EVENT'       : 'O',
'I-FAC'         : 'O',
'I-GPE'         : 'O',
'I-LANGUAGE'    : 'O',
'I-LAW'         : 'O',
'I-LOC'         : 'I-LOC',
'I-MONEY'       : 'O',
'I-NORP'        : 'O',
'I-ORDINAL'     : 'O',
'I-ORG'         : 'I-ORG',
'I-PERCENT'     : 'O',
'I-PERSON'      : 'I-PER',
'I-PRODUCT'     : 'O',
'I-QUANTITY'    : 'O',
'I-TIME'        : 'O',
'I-WORK_OF_ART' : 'O',
'E-CARDINAL'    : 'O',
'E-DATE'        : 'O',
'E-EVENT'       : 'O',
'E-FAC'         : 'O',
'E-GPE'         : 'O',
'E-LANGUAGE'    : 'O',
'E-LAW'         : 'O',
'E-LOC'         : 'E-LOC',
'E-MONEY'       : 'O',
'E-NORP'        : 'O',
'E-ORDINAL'     : 'O',
'E-ORG'         : 'E-ORG',
'E-PERCENT'     : 'O',
'E-PERSON'      : 'E-PER',
'E-PRODUCT'     : 'O',
'E-QUANTITY'    : 'O',
'E-TIME'        : 'O',
'E-WORK_OF_ART' : 'O',
'S-CARDINAL'    : 'O',
'S-DATE'        : 'O',
'S-EVENT'       : 'O',
'S-FAC'         : 'O',
'S-GPE'         : 'O',
'S-LANGUAGE'    : 'O',
'S-LAW'         : 'O',
'S-LOC'         : 'O',
'S-MONEY'       : 'O',
'S-NORP'        : 'O',
'S-ORDINAL'     : 'O',
'S-ORG'         : 'O',
'S-PERCENT'     : 'O',
'S-PERSON'      : 'O',
'S-PRODUCT'     : 'O',
'S-QUANTITY'    : 'O',
'S-TIME'        : 'O',
'S-WORK_OF_ART' : 'O',
}

model_name = "ckiplab/bert-base-chinese-ner"
model_name_output = 'bert-base-chinese-ner'
model_evaluation = ner.ModelEvaluation(
    model_name,
    alignment
)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:72: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [10]:
model_evaluation.model.config.id2label

{0: 'O',
 1: 'B-CARDINAL',
 2: 'B-DATE',
 3: 'B-EVENT',
 4: 'B-FAC',
 5: 'B-GPE',
 6: 'B-LANGUAGE',
 7: 'B-LAW',
 8: 'B-LOC',
 9: 'B-MONEY',
 10: 'B-NORP',
 11: 'B-ORDINAL',
 12: 'B-ORG',
 13: 'B-PERCENT',
 14: 'B-PERSON',
 15: 'B-PRODUCT',
 16: 'B-QUANTITY',
 17: 'B-TIME',
 18: 'B-WORK_OF_ART',
 19: 'I-CARDINAL',
 20: 'I-DATE',
 21: 'I-EVENT',
 22: 'I-FAC',
 23: 'I-GPE',
 24: 'I-LANGUAGE',
 25: 'I-LAW',
 26: 'I-LOC',
 27: 'I-MONEY',
 28: 'I-NORP',
 29: 'I-ORDINAL',
 30: 'I-ORG',
 31: 'I-PERCENT',
 32: 'I-PERSON',
 33: 'I-PRODUCT',
 34: 'I-QUANTITY',
 35: 'I-TIME',
 36: 'I-WORK_OF_ART',
 37: 'E-CARDINAL',
 38: 'E-DATE',
 39: 'E-EVENT',
 40: 'E-FAC',
 41: 'E-GPE',
 42: 'E-LANGUAGE',
 43: 'E-LAW',
 44: 'E-LOC',
 45: 'E-MONEY',
 46: 'E-NORP',
 47: 'E-ORDINAL',
 48: 'E-ORG',
 49: 'E-PERCENT',
 50: 'E-PERSON',
 51: 'E-PRODUCT',
 52: 'E-QUANTITY',
 53: 'E-TIME',
 54: 'E-WORK_OF_ART',
 55: 'S-CARDINAL',
 56: 'S-DATE',
 57: 'S-EVENT',
 58: 'S-FAC',
 59: 'S-GPE',
 60: 'S-LANGUAGE',
 61: 'S-LAW'

### Peoples daily

In [12]:
data_name = "peoples_daily_ner"
pdaily_evaluation_output = model_evaluation.evaluate_model(pdaily_words, pdaily_labels)

  0%|          | 0/290 [00:00<?, ?it/s]

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: Undefin

In [13]:
pdaily_seqeval = pdaily_evaluation_output.get_classification('Seqeval')
pdaily_seqeval

,Tag,Precision,Recall,F1,support
0,LOC,0.7954,0.0855,0.1544,3637
1,ORG,0.7815,0.6892,0.7325,2185
2,PER,0.9395,0.8578,0.8968,1864
3,micro,0.8498,0.4444,0.5836,7686
4,macro,0.8388,0.5442,0.5946,7686
5,weighted,0.8264,0.4444,0.4988,7686


In [14]:
pdaily_sklearn = pdaily_evaluation_output.get_classification('Sklearn')
pdaily_sklearn

,Tag,Precision,Recall,F1,support
0,B-LOC,0.9199,0.0979,0.1769,3637
1,B-ORG,0.8561,0.7245,0.7848,2185
2,B-PER,0.9610,0.8723,0.9145,1864
3,E-LOC,0.0000,0.0000,0.0000,0
4,E-ORG,0.0000,0.0000,0.0000,0
5,E-PER,0.0000,0.0000,0.0000,0
6,I-LOC,0.7632,0.0354,0.0676,4918
7,I-ORG,0.9159,0.6060,0.7294,8756
8,I-PER,0.9683,0.4835,0.6449,3601
9,O,0.9508,0.9963,0.9730,193976


### wikiann

In [15]:
data_name = "wikiann"
wikiann_evaluation_output = model_evaluation.evaluate_model(wikiann_words, wikiann_labels)

  0%|          | 0/625 [00:00<?, ?it/s]

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: Undefin

In [16]:
wikiann_seqeval = wikiann_evaluation_output.get_classification('Seqeval')
wikiann_seqeval

,Tag,Precision,Recall,F1,support
0,LOC,0.4278,0.0536,0.0953,4474
1,ORG,0.3570,0.1699,0.2302,4115
2,PER,0.5217,0.7743,0.6234,3943
3,micro,0.4769,0.3185,0.3820,12532
4,macro,0.4355,0.3326,0.3163,12532
5,weighted,0.4341,0.3185,0.3058,12532


In [17]:
wikiann_sklearn = wikiann_evaluation_output.get_classification('Sklearn')
wikiann_sklearn

,Tag,Precision,Recall,F1,support
0,B-LOC,0.5342,0.0661,0.1177,4371
1,B-ORG,0.4533,0.2173,0.2937,3779
2,B-PER,0.5720,0.8217,0.6745,3899
3,E-LOC,0.0000,0.0000,0.0000,0
4,E-ORG,0.0000,0.0000,0.0000,0
5,E-PER,0.0000,0.0000,0.0000,0
6,I-LOC,0.5935,0.0372,0.0700,12282
7,I-ORG,0.4976,0.1907,0.2757,17399
8,I-PER,0.6179,0.5653,0.5904,12897
9,O,0.8200,0.9244,0.8691,152791


### MRSA

In [18]:
data_name = "msra_ner"
mrsa_evaluation_output = model_evaluation.evaluate_model(mrsa_words, mrsa_labels)

  0%|          | 0/216 [00:00<?, ?it/s]

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: Undefin

In [19]:
mrsa_seqeval = mrsa_evaluation_output.get_classification('Seqeval')
mrsa_seqeval

,Tag,Precision,Recall,F1,support
0,LOC,0.7689,0.0642,0.1184,2852
1,ORG,0.6683,0.6091,0.6373,1320
2,PER,0.9287,0.7716,0.8429,1502
3,micro,0.7981,0.3782,0.5132,5674
4,macro,0.7886,0.4816,0.5329,5674
5,weighted,0.7878,0.3782,0.4309,5674


In [20]:
mrsa_sklearn = mrsa_evaluation_output.get_classification('Sklearn')
mrsa_sklearn

,Tag,Precision,Recall,F1,support
0,B-LOC,0.8889,0.0729,0.1348,2852
1,B-ORG,0.8160,0.6955,0.7509,1320
2,B-PER,0.9697,0.7883,0.8696,1502
3,E-LOC,0.0000,0.0000,0.0000,0
4,E-ORG,0.0000,0.0000,0.0000,0
5,E-PER,0.0000,0.0000,0.0000,0
6,I-LOC,0.8214,0.0316,0.0609,4363
7,I-ORG,0.9020,0.5548,0.6870,5571
8,I-PER,0.9602,0.4451,0.6083,2925
9,O,0.9439,0.9963,0.9694,151719
